In [5]:
# Set Up 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import os
# change working directory
os.chdir('/Users/gerardogutierrez/Desktop/Academics/Spring_2026/plant-health-status/Notebooks')

pd.set_option('display.max_columns', None)  # Show all columns in DataFrame display

# first set of data

In [6]:
SENSOR_DATA_PATH = '../Data/RawData/Trial_1_Sensor_Data.xlsx'

##================## 
# Environmental Data
##================##

# environmental data - night
environmental_data_night = pd.read_excel(SENSOR_DATA_PATH, header=2, usecols="A:H", nrows=119)

# environmental data - day
environmental_data_day = pd.read_excel(SENSOR_DATA_PATH, header=2, usecols="K:R", nrows=236)
# strip ".1" ferom column names
environmental_data_day.columns = environmental_data_day.columns.str.replace('.1', '', regex=False)

# combine night and day and sort by timestamp
environmental_data = pd.concat(
    [environmental_data_night, environmental_data_day],
    axis=0,
    ignore_index=True
)

environmental_data.sort_values("DateTime", inplace=True)
environmental_data.reset_index(drop=True, inplace=True)
environmental_data['DateTime'] = pd.to_datetime(environmental_data['DateTime'])

# rename columns 
environmental_data.rename(columns={
    'Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)': 'Carbon Dioxide',
    'BME280 (CH0, Temperature, °C)': 'Temperature',
    'BME280 (CH1, Humidity, %)': 'Humidity',
    'BME280 (CH2, Pressure, Pa)': 'Pressure',
    'BME280 (CH3, Dewpoint, °C)': 'Dewpoint',
    'BME280 (CH4, Altitude, m)': 'Altitude',
    'BME280 (CH5, Vapor Pressure Deficit, Pa)': 'Vapor Pressure Deficit'
}, inplace=True)

# drop correlated columns and add day column 
def days_by_4(entry):
    if entry == 0:
        return 0
    elif entry in {1,2,3,4}:
        return 4
    elif entry in {5,6,7,8}:
        return 8
    elif entry in {9,10,11,12}:
        return 12
    elif entry in {13,14,15,16}:
        return 16
    elif entry in {17,18,19,20}:
        return 20
    elif entry in {21,22,23,24}:
        return 24
    elif entry in {25,26,27,28}:
        return 28

correlated_col_to_drop = ['Vapor Pressure Deficit', 'Dewpoint', 'Altitude']

env_df = environmental_data.copy()
day0_date = env_df['DateTime'].dt.floor('D').min()
env_df['Temp Days'] = (env_df['DateTime'].dt.floor('D') - day0_date).dt.days
env_df['Day'] = env_df['Temp Days'].apply(days_by_4)
env_df.drop(columns=['Temp Days'], inplace=True)
env_df.drop(columns=correlated_col_to_drop, inplace=True)

# mean dataframe for environmental data 
mean_env_df = (env_df
               .drop(columns=['DateTime'])
               .groupby('Day').mean().reset_index()
)


##=================##
# Nutrient Data 
##=================##
# nutrient data sensor input 
nutrient_data = pd.read_excel(SENSOR_DATA_PATH, sheet_name='Nutrient Data', header=1, usecols = "A:I", nrows=355)

# drop specific gravity column 
nutrient_data.drop(columns=['Atlas EC (CH3, Specific Gravity)'], inplace=True)

# rename columns
nutrient_data.rename(columns={
    'Atlas pH (CH0, Ion Concentration, pH)': 'Ion Concentration',
    'Atlas EC (CH0, Electrical Conductivity, μS/cm)': 'Electrical Conductivity',
    'Atlas EC (CH1, Total Dissolved Solids, ppm)': 'Total Dissolved Solids',
    'Atlas EC (CH2, Salinity, ppt)': 'Salinity',
    'Atlas Flow Meter (CH0, Volume, l)': 'Volume',
    'Atlas PT-1000 (CH0, Temperature, °C)': 'Temperature',
    'Atlas Flow Meter (CH1, Volume Flow Rate, l/min)': 'Volume Flow Rate'
}, inplace=True)
nutrient_data['DateTime'] = pd.to_datetime(nutrient_data['DateTime'])

# drop redundant columns
correlated_col_to_drop = ['Ion Concentration','Salinity', 'Total Dissolved Solids']
nutrient_data.drop(columns=correlated_col_to_drop, inplace=True)


nutr_df = nutrient_data.copy()
nutr_df['Temp Days'] = (nutr_df['DateTime'].dt.floor('D') - day0_date).dt.days
nutr_df['Day'] = nutr_df['Temp Days'].apply(days_by_4)
nutr_df.head(30)
nutr_df.drop(columns=[ 'Temp Days'], inplace=True)

# mean dataframe for nutrient data 
mean_nutr_df = (nutr_df
                .drop(columns=['DateTime'])
                .groupby('Day').mean().reset_index()
)



In [7]:
import pandas as pd

# function to load lettuce weight data into dataframe
def load_lettuce_weights(file_path='../Data/RawData/Trial_1_Lettuce_FW.xlsx'):
    weights_df = pd.read_excel(file_path, sheet_name='Data_Collection', header=0)
    weights_df = weights_df.drop(columns=['Date'])
    weights_df = weights_df.dropna(how='all')
    return weights_df

def load_mean_nutr_env_df():
    """Join mean environmental and nutrient data on 'Days' column."""
    return pd.merge(mean_env_df.rename(columns={'Temperature': 'Temp (env)'}), mean_nutr_df.rename(columns={'Temperature': 'Temp (nutr)'}), on='Day')


In [8]:

model_df = (load_lettuce_weights().merge(load_mean_nutr_env_df(), on='Day')
          .drop(columns=['Median Fresh Weight (g)', 'Average Fresh Weight (g)', 'New Fresh Weight (g)'])
)

target = "Total Fresh Weight (g)"

feature_cols = [
    "Carbon Dioxide",
    "Temp (env)",
    "Humidity",
    "Pressure",
    "Electrical Conductivity",
    "Volume",
    "Temp (nutr)",
    "Volume Flow Rate"
]

for lag in [1, 2]:
        model_df[f"weight_lag_{lag}"] = (
        model_df.groupby("Plant-ID")[target].shift(lag)
    )


# rename all columns to lower case, remove parentheses  and replace spaces with snake case
model_df.columns = (model_df.columns.str.lower()
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace(' ', '_')
)

model_df.drop(columns='baseline_g')

model_df.head(30)

,day,plant-id,total_fresh_weight_g,baseline_g,carbon_dioxide,temp_env,humidity,pressure,electrical_conductivity,volume,temp_nutr,volume_flow_rate,weight_lag_1,weight_lag_2
0,0.0,6.0,29.1,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
1,0.0,4.0,29.3,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
2,0.0,9.0,29.6,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
3,0.0,1.0,29.7,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
4,0.0,8.0,29.8,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
5,0.0,2.0,30.1,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
6,0.0,5.0,30.2,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
7,0.0,7.0,30.4,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
8,0.0,10.0,31.1,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN
9,0.0,3.0,31.8,25.22,461.25000,22.528333,62.394167,98838.439167,1300.625000,1010.277917,23.835208,1.160000,NaN,NaN


save 3 dataframes as csv 
* no lag 
* with lag 1 
* with lag 2 

In [9]:

TRIAL_1_MEAN_SENSOR_NO_LAG_DATA = model_df.drop(columns=['weight_lag_1', 'weight_lag_2'])
TRIAL_1_MEAN_SENSOR_LAG_1_DATA = model_df.drop(columns=['weight_lag_2']).dropna(subset=['weight_lag_1'])
TRIAL_1_MEAN_SENSOR_LAG_2_DATA = model_df.copy().dropna(subset=['weight_lag_1', 'weight_lag_2'])

In [10]:
# save dfs as csv
TRIAL_1_MEAN_SENSOR_NO_LAG_DATA.to_csv('../Data/CleanData/Trial_1_Mean_Sensor_FW_No_Lag.csv', index=False)
TRIAL_1_MEAN_SENSOR_LAG_1_DATA.to_csv('../Data/CleanData/Trial_1_Mean_Sensor_FW_Lag_1.csv', index=False)
TRIAL_1_MEAN_SENSOR_LAG_2_DATA.to_csv('../Data/CleanData/Trial_1_Mean_Sensor_FW_Lag_2.csv', index=False)

# second set of data 